# 3. Building a Support Knowledge Graph with an LLM: Extraction, Entity Resolution, Validation, Provenance

**Presentation map:** Slide 46 (schema-first vs. schema-light construction), Slide 47 (extraction → canonicalisation →
validation pipeline), Slide 48 (entity resolution as a cost cascade), Slide 49 (identity, provenance, time and review
status on every assertion), Slide 50 (structural validity is not factual truth)

**Dataset:** `data/planetcare_tickets.json` (183 real PlanetCare EHR support tickets, 7+ hospitals),
`data/planetcare_kb.json` (3 existing KB articles) and `data/planetcare_wiki/billing.md` (the team's billing wiki,
including its hand-written "Knowledge Gaps" list).

The construction pipeline, one stage per section:

1. **Entity resolution** of the messy `module` field (62 spellings) with a cheap-first cascade.
2. **Schema-light vs. schema-first extraction** of issue patterns from every billing ticket, then EDC-style
   canonicalisation.
3. **Validation** — deterministic shape checks *and* a separate source-support check.
4. **Provenance-carrying assertions** and a KB article drafted only from validated, cited assertions.

In [1]:
import json, re, itertools
from collections import Counter, defaultdict
import networkx as nx

tickets = json.load(open("data/planetcare_tickets.json"))
kb_articles = json.load(open("data/planetcare_kb.json"))
T = {t["ticket_id"]: t for t in tickets}

G = nx.DiGraph()
for t in tickets:
    G.add_node(t["ticket_id"], label="Ticket", **t)
    G.add_node(("Hospital", t["hospital"]), label="Hospital")
    G.add_edge(t["ticket_id"], ("Hospital", t["hospital"]), type="REPORTED_BY")
    G.add_node(("ModuleMention", t["module"]), label="ModuleMention")
    G.add_edge(t["ticket_id"], ("ModuleMention", t["module"]), type="IN_MODULE")

raw_modules = Counter(t["module"] for t in tickets)
print(f"{len(tickets)} tickets ({Counter(t['status'] for t in tickets)}), {len(kb_articles)} KB articles, "
      f"{len({t['hospital'] for t in tickets})} hospitals")
print(f"Distinct raw module strings: {len(raw_modules)}")
print("Examples of the same module under different names:",
      [m for m in raw_modules if re.search(r"bill|wait|order|theatre|lab|admin|referral", m, re.I)])

183 tickets (Counter({'Open': 122, 'Closed': 61})), 3 KB articles, 10 hospitals
Distinct raw module strings: 62
Examples of the same module under different names: ['Laboratory Management', 'Order Management', 'Theatre Management', 'Billing Management', 'Waiting List Management', 'Patient Administration', 'Referral Management', 'Billing', 'Waiting Lists', 'Laboratory', 'Orders Management', 'Operating Theatres', 'Patient Admin', 'Theatres', 'Orders', 'E-referral', 'E-Referral', 'Waiting List', 'Administration']


## Stage 1 — Entity resolution as a cost cascade (Slide 48)

Cheap and certain first, LLM last: **(1)** exact match on a normalised key (case, hyphens, plurals, the generic word
"Management") → **(2)** embedding similarity: auto-merge above a high threshold, discard below a low one →
**(3)** LLM adjudication *only* for the ambiguous band in between → **(4)** low-confidence LLM verdicts go to a human
review queue instead of being merged.

In [2]:
import os
import numpy as np
from typing import Literal, Optional
from concurrent.futures import ThreadPoolExecutor
from pydantic import BaseModel, Field
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
CHAT_MODEL, EMBED_MODEL = "gpt-4o-mini", "text-embedding-3-small"
AUTO_MERGE, AMBIGUOUS = 0.85, 0.55            # cosine thresholds for stage 2

def parse(model, system, user):
    return client.beta.chat.completions.parse(model=CHAT_MODEL, temperature=0, seed=7, response_format=model,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}]).choices[0].message.parsed

parent = {m: m for m in raw_modules}             # union-find over module names
def find(x):
    while parent[x] != x:
        x = parent[x]
    return x
def union(a, b):
    parent[find(b)] = find(a)

# Stage 1: exact match on a normalised key
def norm_key(name):
    words = [w for w in re.sub(r"[-_]", " ", name.lower()).split() if w != "management"]
    return " ".join(w[:-1] if w.endswith("s") and len(w) > 3 and not w.endswith("ss") else w for w in words)
by_key = defaultdict(list)
for m in raw_modules:
    by_key[norm_key(m)].append(m)
stage1 = 0
for members in by_key.values():
    for m in members[1:]:
        union(members[0], m); stage1 += 1

# Stage 2: embedding similarity between the remaining clusters (one representative name each)
reps = sorted({find(m) for m in raw_modules})
E = np.array([e.embedding for e in client.embeddings.create(model=EMBED_MODEL, input=reps).data])
E /= np.linalg.norm(E, axis=1, keepdims=True)
S = E @ E.T
all_pairs = list(itertools.combinations(range(len(reps)), 2))
auto = [(reps[i], reps[j], S[i, j]) for i, j in all_pairs if S[i, j] >= AUTO_MERGE]
ambiguous = [(reps[i], reps[j], S[i, j]) for i, j in all_pairs if AMBIGUOUS <= S[i, j] < AUTO_MERGE]
for a, b, _ in auto:
    union(a, b)

# Stage 3: LLM adjudication only for the ambiguous band
class SameEntity(BaseModel):
    same_entity: bool
    confidence: Literal["high", "low"]
    reason: str

def adjudicate(pair):
    a, b, s = pair
    return a, b, s, parse(SameEntity,
        "You resolve entities in a hospital EHR support system. Decide whether two module names written by different "
        "support teams refer to the SAME functional module of the EHR product. Related-but-different modules are NOT the same.",
        f"Module A: {a!r}\nModule B: {b!r}")
with ThreadPoolExecutor(8) as ex:
    verdicts = list(ex.map(adjudicate, ambiguous))
review_queue = []
for a, b, s, v in verdicts:
    if v.same_entity and v.confidence == "high":
        union(a, b)
    elif v.same_entity:
        review_queue.append((a, b, round(float(s), 3), v.reason))   # Stage 4: human review, not merged

n_pairs = len(raw_modules) * (len(raw_modules) - 1) // 2
llm_merges = sum(v.same_entity and v.confidence == "high" for *_, v in verdicts)
print(f"Candidate pairs if every pair went to the LLM : {n_pairs:,}")
print(f"Stage 1 exact normalised key  : {stage1} merges (free, deterministic)")
print(f"Stage 2 embedding >= {AUTO_MERGE}      : {len(auto)} merges {[(a, b) for a, b, _ in auto]}")
print(f"        embedding <  {AMBIGUOUS}      : {len(all_pairs) - len(auto) - len(ambiguous):,} pairs discarded without an LLM call")
print(f"Stage 3 LLM adjudication      : {len(ambiguous)} calls -> {llm_merges} merges   "
      f"({100 * (1 - len(ambiguous) / n_pairs):.1f}% of pairwise LLM calls saved)")
print(f"Stage 4 human review queue    : {len(review_queue)} low-confidence pairs")
for a, b, s, v in sorted(verdicts, key=lambda x: -x[2])[:12]:
    print(f"   {s:.2f}  {a!r:<26} ~ {b!r:<28} same={v.same_entity!s:<5} ({v.confidence})")

# Pairwise verdicts are not transitive: flag clusters that contain a pair the LLM explicitly called different
conflicts = [(a, b) for a, b, s, v in verdicts if not v.same_entity and find(a) == find(b)]
review_queue += [(a, b, None, "transitivity conflict") for a, b in conflicts]
print(f"\nTransitivity conflicts (merged via a third name, but judged different pairwise): {conflicts} -> sent to review")

Candidate pairs if every pair went to the LLM : 1,891
Stage 1 exact normalised key  : 11 merges (free, deterministic)
Stage 2 embedding >= 0.85      : 1 merges [('Patient Admin', 'Patient Administration')]
        embedding <  0.55      : 1,229 pairs discarded without an LLM call
Stage 3 LLM adjudication      : 45 calls -> 5 merges   (97.6% of pairwise LLM calls saved)
Stage 4 human review queue    : 0 low-confidence pairs
   0.81  'Reporting'                ~ 'Reports'                    same=False (high)
   0.80  'User Maintenance'         ~ 'User Management'            same=True  (high)
   0.79  'EHR'                      ~ 'EMR'                        same=False (high)
   0.74  'Medical Records'          ~ 'Patient Records'            same=False (high)
   0.74  'EMR'                      ~ 'Electronic Patient Record'  same=True  (high)
   0.73  'Patient Administration'   ~ 'Patient Management'         same=False (high)
   0.71  'EHR'                      ~ 'Electronic Patient Recor

In [3]:
# Rewire the graph: every ModuleMention -> canonical Module (named after its most-used spelling)
clusters = defaultdict(list)
for m in raw_modules:
    clusters[find(m)].append(m)
canonical = {}
for members in clusters.values():
    name = max(members, key=lambda m: raw_modules[m])
    for m in members:
        canonical[m] = name
for m, c in canonical.items():
    G.add_node(("Module", c), label="Module")
    G.add_edge(("ModuleMention", m), ("Module", c), type="RESOLVED_TO")

after = Counter(canonical[t["module"]] for t in tickets)
print(f"Modules: {len(raw_modules)} raw strings -> {len(after)} canonical modules\n")
print("Merged groups:")
for c, members in sorted(((canonical[ms[0]], ms) for ms in clusters.values() if len(ms) > 1), key=lambda x: -after[x[0]]):
    print(f"  {c:<26} <- {sorted(members)}")
print(f"\n{'Top modules':<26} {'before ER':>9} {'after ER':>9}")
for c, n in after.most_common(8):
    print(f"  {c:<24} {raw_modules.get(c, 0):>9} {n:>9}")

Modules: 62 raw strings -> 45 canonical modules

Merged groups:
  EMR                        <- ['EHR', 'EMR', 'Electronic Patient Record']
  Billing                    <- ['Billing', 'Billing Management']
  System                     <- ['System', 'System Management']
  Waiting Lists              <- ['Waiting List', 'Waiting List Management', 'Waiting Lists']
  Patient Administration     <- ['Patient Admin', 'Patient Administration']
  Laboratory Management      <- ['Laboratory', 'Laboratory Management']
  User Management            <- ['User Maintenance', 'User Management']
  Orders                     <- ['Order Management', 'Orders', 'Orders Management']
  Theatre Management         <- ['Operating Theatres', 'Theatre Management', 'Theatres']
  Radiology                  <- ['Radiology', 'Radiology Management']
  Coding                     <- ['Coding', 'Coding Management']
  Configuration              <- ['Configuration', 'System Configuration']
  E-referral                 <- ['E-

## Stage 2 — Schema-light vs. schema-first extraction, then canonicalisation (Slides 46-47)

We take **every** ticket classified `BILLING` or `REVENUE_CYCLE` (the classification, not the module field — one billing
ticket is filed under `EMR`). Each ticket is extracted independently, as a streaming pipeline would.

- **Schema-light:** free-text `issue_pattern`. Expect fragmentation — every ticket invents its own label.
- **Schema-first:** the pattern must come from a fixed list taken from the team's own wiki (two existing KB topics + its
  three documented knowledge gaps) plus `Other`. Precise, but it misses what the schema did not anticipate.
- **EDC canonicalisation** (Extract → Define → Canonicalise): define each free-text label, induce a small canonical set,
  and map every label onto it.

In [4]:
billing = [t for t in tickets if t["classification"] in ("BILLING", "REVENUE_CYCLE")]
src = lambda t: (f"{t['ticket_id']} [{t['status']}] summary: {t['summary']} | description: {t['description']} | "
                 f"resolution: {t['resolution'] or '(none - ticket open)'}")
print(f"Billing cluster: {len(billing)} tickets ({Counter(t['status'] for t in billing)}) across "
      f"{len({t['hospital'] for t in billing})} hospitals; modules: {dict(Counter(t['module'] for t in billing))}")

class OpenExtraction(BaseModel):                       # schema-light
    issue_pattern: str = Field(description="Short reusable name for the underlying issue")
    affected_component: str = Field(description="Billing subsystem affected")
    root_cause: Optional[str] = Field(description="Root cause, only if stated in the ticket, else null")
    verified_fix: Optional[str] = Field(description="Fix that was applied, only if the ticket is resolved, else null")

WIKI_PATTERNS = ("Claim rejection", "Invoice not printing", "Discount amount mismatch",
                 "Batch billing error", "Insurance payment delay", "Other")
class SchemaFirst(BaseModel):                          # schema-first
    issue_pattern: Literal[WIKI_PATTERNS]

EXTRACT_SYS = "You extract reusable support-knowledge entities from one raw hospital billing support ticket."
with ThreadPoolExecutor(8) as ex:
    light = dict(zip([t["ticket_id"] for t in billing], ex.map(lambda t: parse(OpenExtraction, EXTRACT_SYS, src(t)), billing)))
    first = dict(zip([t["ticket_id"] for t in billing], ex.map(lambda t: parse(SchemaFirst, EXTRACT_SYS, src(t)), billing)))

raw_labels = Counter(x.issue_pattern for x in light.values())
sf = Counter(x.issue_pattern for x in first.values())
print(f"\nSchema-light: {len(raw_labels)} distinct pattern labels for {len(billing)} tickets:")
print("  ", dict(raw_labels.most_common()))
print(f"Schema-first: {dict(sf.most_common())}  -> {sf['Other']}/{len(billing)} tickets fall outside the schema")

Billing cluster: 29 tickets (Counter({'Open': 16, 'Closed': 13})) across 9 hospitals; modules: {'Billing Management': 1, 'Billing': 22, 'Revenue Cycle': 3, 'Finance': 1, 'EMR': 2}



Schema-light: 19 distinct pattern labels for 29 tickets:
   {'Billing Discrepancy': 8, 'Billing Inquiry': 3, 'Billing Query': 2, 'Billing Error': 1, 'Billing statement generation issue': 1, 'Revenue cycle discrepancies': 1, 'Billing clarification': 1, 'Billing dispute for specialized treatment': 1, 'Billing discrepancy': 1, 'Insurance Verification Issue': 1, 'Duplicate Billing': 1, 'Revenue cycle management query': 1, 'Revenue Cycle Reporting Issue': 1, 'Billing Statement Not Received': 1, 'Billing System Performance Issue': 1, 'Billing clarification request': 1, 'Billing statement not received': 1, 'Billing Cycle Issues': 1, 'Revenue Cycle Inquiry': 1}
Schema-first: {'Other': 24, 'Discount amount mismatch': 3, 'Claim rejection': 1, 'Invoice not printing': 1}  -> 24/29 tickets fall outside the schema


In [5]:
# EDC: Define each raw label, induce a canonical set, Canonicalise every label onto it
class Definition(BaseModel):
    label: str
    definition: str
class Definitions(BaseModel):
    items: list[Definition]
class CanonicalPattern(BaseModel):
    name: str
    definition: str
class CanonicalSet(BaseModel):
    patterns: list[CanonicalPattern] = Field(description="3 to 6 mutually exclusive canonical issue patterns")

defs = parse(Definitions, "Write a one-sentence definition for each billing issue label.", "\n".join(raw_labels)).items
canon = parse(CanonicalSet, "Induce a small set of canonical, mutually exclusive billing issue patterns that covers all "
              "the defined labels. Merge labels that describe the same underlying issue.",
              "\n".join(f"- {d.label}: {d.definition}" for d in defs)).patterns
CANON = tuple(p.name for p in canon)

class Mapping(BaseModel):
    canonical: Literal[CANON]
with ThreadPoolExecutor(8) as ex:
    label_map = dict(zip(raw_labels, ex.map(lambda l: parse(Mapping,
        "Map the billing issue label to exactly one canonical pattern.\n" +
        "\n".join(f"- {p.name}: {p.definition}" for p in canon), l).canonical, raw_labels)))

pattern_of = {tid: label_map[x.issue_pattern] for tid, x in light.items()}
print(f"Canonicalisation: {len(raw_labels)} free-text labels -> {len(CANON)} canonical patterns\n")
for p in canon:
    members = [tid for tid, c in pattern_of.items() if c == p.name]
    closed = [tid for tid in members if T[tid]["status"] == "Closed"]
    print(f"  {p.name:<34} {len(members):>2} tickets ({len(closed)} closed, {len({T[t]['hospital'] for t in members})} hospitals)")
    print(f"      from labels: {sorted({light[t].issue_pattern for t in members})}")

Canonicalisation: 19 free-text labels -> 6 canonical patterns

  Billing Discrepancy                10 tickets (5 closed, 5 hospitals)
      from labels: ['Billing Discrepancy', 'Billing discrepancy', 'Billing dispute for specialized treatment']
  Billing Error                       2 tickets (2 closed, 2 hospitals)
      from labels: ['Billing Error', 'Duplicate Billing']
  Billing Statement Issues            3 tickets (1 closed, 3 hospitals)
      from labels: ['Billing Statement Not Received', 'Billing statement generation issue', 'Billing statement not received']
  Insurance Verification Issues       1 tickets (0 closed, 1 hospitals)
      from labels: ['Insurance Verification Issue']
  Revenue Cycle Issues                6 tickets (3 closed, 4 hospitals)
      from labels: ['Billing Cycle Issues', 'Billing System Performance Issue', 'Revenue Cycle Inquiry', 'Revenue Cycle Reporting Issue', 'Revenue cycle discrepancies', 'Revenue cycle management query']
  Billing Inquiry and Clari

## Stage 3 — Validation: shape checks vs. source support (Slide 50)

Every extracted fact becomes a candidate assertion. Two **independent** checks:

- **Structural (SHACL-style, deterministic):** known subject, allowed predicate, object of the right type, a
  `HAS_FIX` only on a `Closed` ticket, provenance present.
- **Source support (LLM judge, separate):** does the ticket text actually say this?

In [6]:
from datetime import datetime, timezone
RECORDED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
EXTRACTOR, PROMPT_VERSION = f"openai:{CHAT_MODEL}", "billing-open-extraction-v1"

assertions = []
for tid, x in light.items():
    for pred, obj, otype in (("EXHIBITS_PATTERN", pattern_of[tid], "IssuePattern"),
                             ("HAS_ROOT_CAUSE", x.root_cause, "RootCause"),
                             ("HAS_FIX", x.verified_fix, "Fix")):
        if obj:
            assertions.append({
                "assertion_id": f"assertion:{len(assertions) + 1:04d}",
                "subject": {"entity_id": tid, "entity_type": "Ticket"},
                "predicate": pred,
                "object": {"value": obj, "entity_type": otype},
                "provenance": {"document_id": "planetcare_tickets.json", "passage_id": tid,
                               "extractor_version": EXTRACTOR, "prompt_version": PROMPT_VERSION},
                "valid_from": None, "valid_to": None,           # tickets carry no event time in this dataset
                "recorded_at": RECORDED_AT, "superseded_by": None,
                "validation": {}})

ALLOWED = {"EXHIBITS_PATTERN": "IssuePattern", "HAS_ROOT_CAUSE": "RootCause", "HAS_FIX": "Fix"}
def shape_check(a):
    tid = a["subject"]["entity_id"]
    errors = []
    if G.nodes.get(tid, {}).get("label") != "Ticket":            errors.append("unknown subject")
    if ALLOWED.get(a["predicate"]) != a["object"]["entity_type"]: errors.append("predicate/object type mismatch")
    if a["predicate"] == "EXHIBITS_PATTERN" and a["object"]["value"] not in CANON: errors.append("pattern not canonical")
    if a["predicate"] == "HAS_FIX" and T[tid]["status"] != "Closed": errors.append("fix asserted on an open ticket")
    if not all(a["provenance"].values()):                         errors.append("missing provenance")
    return errors

class Support(BaseModel):
    supported: bool = Field(description="True only if the ticket text explicitly states or directly implies the claim")
    evidence: str = Field(description="Quote from the ticket, or empty")
def support_check(a):
    claim = f"{a['predicate']}: {a['object']['value']}"
    return parse(Support, "You verify extracted claims against their source. Be strict: details that are not in the "
                 "ticket (causes, systems, actions) make the claim unsupported.",
                 f"Ticket: {src(T[a['subject']['entity_id']])}\n\nClaim about this ticket: {claim}")

with ThreadPoolExecutor(8) as ex:
    supports = list(ex.map(support_check, assertions))
for a, s in zip(assertions, supports):
    errs = shape_check(a)
    a["validation"] = {"schema_status": "passed" if not errs else "failed: " + "; ".join(errs),
                       "source_support_status": "supported" if s.supported else "unsupported",
                       "entity_resolution_status": "accepted", "human_reviewed": False}

print(f"{'predicate':<18} {'n':>3} {'shape pass':>11} {'source-supported':>17} {'shape pass BUT unsupported':>27}")
for pred in ALLOWED:
    sub = [a for a in assertions if a["predicate"] == pred]
    ok = [a for a in sub if a["validation"]["schema_status"] == "passed"]
    sup = [a for a in sub if a["validation"]["source_support_status"] == "supported"]
    gap = [a for a in ok if a["validation"]["source_support_status"] == "unsupported"]
    print(f"{pred:<18} {len(sub):>3} {len(ok):>11} {len(sup):>17} {len(gap):>27}")

print("\nStructurally valid but NOT supported by the source (examples):")
for a in [a for a in assertions if a["validation"]["schema_status"] == "passed"
          and a["validation"]["source_support_status"] == "unsupported"][:5]:
    t = T[a["subject"]["entity_id"]]
    print(f"  {a['subject']['entity_id']} {a['predicate']} -> {a['object']['value']!r}")
    print(f"      source says: {t['summary']!r} / resolution: {t['resolution'] or '(none)'!r}")
print("\nShape failures:", Counter(a["validation"]["schema_status"] for a in assertions if a["validation"]["schema_status"] != "passed"))

predicate            n  shape pass  source-supported  shape pass BUT unsupported
EXHIBITS_PATTERN    29          29                26                           3
HAS_ROOT_CAUSE       2           2                 2                           0
HAS_FIX             13          13                13                           0

Structurally valid but NOT supported by the source (examples):
  PC-00134 EXHIBITS_PATTERN -> 'Revenue Cycle Issues'
      source says: 'Revenue cycle management query' / resolution: 'Clarified processes with the team.'
  PC-00004 EXHIBITS_PATTERN -> 'Revenue Cycle Issues'
      source says: 'Billing System Performance Issue' / resolution: '(none)'
  PC-00137 EXHIBITS_PATTERN -> 'Revenue Cycle Issues'
      source says: 'Revenue Cycle Inquiry' / resolution: '(none)'

Shape failures: Counter()


## Stage 4 — Provenance-carrying assertions and a cited KB draft (Slide 49)

Only assertions that pass **both** checks enter the graph. The KB article for the largest canonical pattern is drafted
from them, every step must cite ticket IDs, and the citations are checked deterministically. The draft stays
`human_reviewed: false` until a person approves it.

In [7]:
accepted = [a for a in assertions if a["validation"]["schema_status"] == "passed"
            and a["validation"]["source_support_status"] == "supported"]
for a in accepted:
    o = (a["object"]["entity_type"], a["object"]["value"])
    G.add_node(o, label=a["object"]["entity_type"])
    G.add_edge(a["subject"]["entity_id"], o, type=a["predicate"], assertion_id=a["assertion_id"])
print(f"Accepted {len(accepted)}/{len(assertions)} assertions into the graph. Example record:")
print(json.dumps(next(a for a in accepted if a["predicate"] == "HAS_FIX"), indent=2))

# Largest canonical pattern that has validated fixes
fixes = defaultdict(list)
for a in accepted:
    if a["predicate"] == "HAS_FIX":
        fixes[pattern_of[a["subject"]["entity_id"]]].append(a)
target = max(CANON, key=lambda p: (len(fixes[p]) > 0, sum(c == p for c in pattern_of.values())))
members = [tid for tid, c in pattern_of.items() if c == target]

class Step(BaseModel):
    text: str
    cited_tickets: list[str]
class DraftKB(BaseModel):
    title: str
    problem_statement: str
    resolution_steps: list[Step]

wiki = open("data/planetcare_wiki/billing.md").read()
evidence = "\n".join(f"- {a['subject']['entity_id']}: {a['predicate']} = {a['object']['value']}  [{a['assertion_id']}]"
                     for a in accepted if a["subject"]["entity_id"] in members)
draft = parse(DraftKB, "You write PlanetCare KB articles strictly from the validated assertions given. Every step must "
              "cite the ticket IDs whose assertions support it. Do not add facts that are not in the assertions.",
              f"Write KB-PC-0003 for the issue pattern '{target}'. Match the style of this wiki:\n{wiki[:900]}\n\n"
              f"Validated assertions:\n{evidence}")

cited = {c for s in draft.resolution_steps for c in s.cited_tickets}
bad = cited - set(members)
print(f"\n--- KB-PC-0003: {draft.title.replace('KB-PC-0003:', '').strip()}  (pattern '{target}', {len(members)} tickets) ---")
print(draft.problem_statement)
for i, s in enumerate(draft.resolution_steps, 1):
    print(f"  {i}. {s.text}  [{', '.join(s.cited_tickets)}]")
print(f"Citation check: {len(cited)} tickets cited, {len(bad)} outside the cluster {sorted(bad) if bad else ''}; "
      f"status: requires_human_review=True")
G.add_node("KB-PC-0003", label="KBArticle", draft=True, human_reviewed=False, title=draft.title)
for tid in cited & set(members):
    G.add_edge("KB-PC-0003", tid, type="DERIVED_FROM")

Accepted 41/44 assertions into the graph. Example record:
{
  "assertion_id": "assertion:0002",
  "subject": {
    "entity_id": "PC-00010",
    "entity_type": "Ticket"
  },
  "predicate": "HAS_FIX",
  "object": {
    "value": "Billing issue resolved and corrected.",
    "entity_type": "Fix"
  },
  "provenance": {
    "document_id": "planetcare_tickets.json",
    "passage_id": "PC-00010",
    "extractor_version": "openai:gpt-4o-mini",
    "prompt_version": "billing-open-extraction-v1"
  },
  "valid_from": null,
  "valid_to": null,
  "recorded_at": "2026-10-17T03:17:26+00:00",
  "superseded_by": null,
  "validation": {
    "schema_status": "passed",
    "source_support_status": "supported",
    "entity_resolution_status": "accepted",
    "human_reviewed": false
  }
}



--- KB-PC-0003: Resolving Billing Discrepancies  (pattern 'Billing Discrepancy', 10 tickets) ---
Billing discrepancies identified in patient accounts leading to incorrect charges or refunds.
  1. Identify the billing discrepancy in the patient's account.  [PC-00010, PC-00006, PC-00020, PC-00018, PC-00003, PC-00021, PC-00023, PC-00022, PC-00017]
  2. Review the details of the discrepancy to determine the cause (e.g., incorrect charges, missing information).  [PC-00010, PC-00006, PC-00020, PC-00018, PC-00003, PC-00021, PC-00023, PC-00022, PC-00017]
  3. Correct the billing information as necessary to resolve the discrepancy.  [PC-00010, PC-00018, PC-00022, PC-00017]
  4. Notify the patient of the correction and any adjustments made to their account.  [PC-00018, PC-00003, PC-00022, PC-00017]
  5. Confirm that the billing issue has been resolved and the patient is satisfied with the outcome.  [PC-00010, PC-00018, PC-00022, PC-00017]
Citation check: 9 tickets cited, 0 outside the cluster ;

In [8]:
n_struct = sum(a["validation"]["schema_status"] == "passed" for a in assertions)
n_sup = sum(a["validation"]["source_support_status"] == "supported" for a in assertions)
n_gap = sum(a["validation"]["schema_status"] == "passed" and a["validation"]["source_support_status"] == "unsupported"
            for a in assertions)
print("=" * 72)
print("RESULTS SUMMARY — Notebook 3: an LLM builds a support knowledge graph")
print("=" * 72)
print(f"Entity resolution  : {len(raw_modules)} module strings -> {len(after)} modules | stage1 exact {stage1}, "
      f"stage2 embedding {len(auto)}, stage3 LLM {llm_merges} merges from {len(ambiguous)} calls, "
      f"review queue {len(review_queue)} ({len(conflicts)} transitivity conflicts)")
print(f"LLM calls saved    : {len(ambiguous)} calls instead of {n_pairs:,} pairwise ({100 * (1 - len(ambiguous) / n_pairs):.1f}% saved)")
print(f"Billing module     : {raw_modules['Billing']} tickets before ER -> {after[canonical['Billing']]} after")
print(f"Extraction         : {len(billing)} billing tickets | schema-light {len(raw_labels)} distinct labels -> "
      f"EDC {len(CANON)} canonical | schema-first 'Other' = {sf['Other']}/{len(billing)}")
print(f"Validation         : {len(assertions)} assertions | shape-valid {n_struct} ({100*n_struct/len(assertions):.0f}%) | "
      f"source-supported {n_sup} ({100*n_sup/len(assertions):.0f}%) | shape-valid but unsupported {n_gap}")
print(f"KB-PC-0003         : '{target}', {len(draft.resolution_steps)} steps, {len(cited)} cited tickets, "
      f"{len(bad)} invalid citations, human_reviewed=False")
print("\nLecture message: construction is where the cost and the errors live (Slides 45-48): open extraction fragments,")
print("schema-first misses the unanticipated, entity resolution is a cheap-first cascade, and passing the shape check")
print("is not the same as being true (Slide 50) — so every assertion carries provenance and review status (Slide 49).")

RESULTS SUMMARY — Notebook 3: an LLM builds a support knowledge graph
Entity resolution  : 62 module strings -> 45 modules | stage1 exact 11, stage2 embedding 1, stage3 LLM 5 merges from 45 calls, review queue 1 (1 transitivity conflicts)
LLM calls saved    : 45 calls instead of 1,891 pairwise (97.6% saved)
Billing module     : 22 tickets before ER -> 23 after
Extraction         : 29 billing tickets | schema-light 19 distinct labels -> EDC 6 canonical | schema-first 'Other' = 24/29
Validation         : 44 assertions | shape-valid 44 (100%) | source-supported 41 (93%) | shape-valid but unsupported 3
KB-PC-0003         : 'Billing Discrepancy', 5 steps, 9 cited tickets, 0 invalid citations, human_reviewed=False

Lecture message: construction is where the cost and the errors live (Slides 45-48): open extraction fragments,
schema-first misses the unanticipated, entity resolution is a cheap-first cascade, and passing the shape check
is not the same as being true (Slide 50) — so every asserti